# Experiment 2: Inversion Results

Computes accuracy for:
- **LIT / Llama3 inverter** (zero-shot on inversion outputs)
- **Patchscopes / T5 inverter** (zero-shot on inversion outputs)

In [6]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

TASKS = [
    "country_currency",
    "food_from_country",
    "person_plays_position_in_sport",
    "person_plays_pro_sport",
    "product_by_company",
    "star_constellation",
]

INV_PATH = Path("/home/li.mil/verb_faithfulness/results/inversion")

LIT_DIR = INV_PATH / "llama3_inversion_llama3_multi_zeroshot"
T5_DIR  = INV_PATH / "llama3_inversion_t5-base_single_zeroshot"

In [7]:
def load_results(path):
    if not path.exists():
        return None
    with open(path) as f:
        return json.load(f)

def check_correct(completion, answer):
    if not completion or not answer:
        return False
    return answer.lower() in completion.lower()

def compute_accuracy(results):
    if not results:
        return None
    correct = sum(check_correct(v['completion'], v['answer']) for v in results.values())
    return correct / len(results)

In [8]:
methods = {
    "LIT / Llama3":       LIT_DIR,
    "Patchscopes / T5":   T5_DIR,
}

acc = {}
for method, base_dir in methods.items():
    acc[method] = {}
    for task in TASKS:
        results = load_results(base_dir / f"{task}.json")
        acc[method][task] = compute_accuracy(results)

df = pd.DataFrame(acc).T
df["AVERAGE"] = df.mean(axis=1)
df.round(4)

,country_currency,food_from_country,person_plays_position_in_sport,person_plays_pro_sport,product_by_company,star_constellation,AVERAGE
LIT / Llama3,0.8203,0.5909,0.5755,0.7557,0.6770,0.4185,0.6396
Patchscopes / T5,0.4844,0.2045,0.2270,0.5422,0.3895,0.0326,0.3134


## Per-task breakdown

In [9]:
for method in methods:
    print(f"\n{method}")
    print("=" * 55)
    for task in TASKS:
        a = acc[method][task]
        val = f"{a:.4f}" if a is not None else "missing"
        print(f"  {task:40s}: {val}")
    valid = [a for a in acc[method].values() if a is not None]
    print(f"  {'AVERAGE':40s}: {np.mean(valid):.4f}" if valid else "  No results")


LIT / Llama3
  country_currency                        : 0.8203
  food_from_country                       : 0.5909
  person_plays_position_in_sport          : 0.5755
  person_plays_pro_sport                  : 0.7557
  product_by_company                      : 0.6770
  star_constellation                      : 0.4185
  AVERAGE                                 : 0.6396

Patchscopes / T5
  country_currency                        : 0.4844
  food_from_country                       : 0.2045
  person_plays_position_in_sport          : 0.2270
  person_plays_pro_sport                  : 0.5422
  product_by_company                      : 0.3895
  star_constellation                      : 0.0326
  AVERAGE                                 : 0.3134


## Inspect individual examples

In [10]:
INSPECT_METHOD = "LIT / Llama3"   # or "Patchscopes / T5"
INSPECT_TASK   = "country_currency"
N_SHOW         = 10

results = load_results(methods[INSPECT_METHOD] / f"{INSPECT_TASK}.json")
if results:
    for i, (k, v) in enumerate(results.items()):
        if i >= N_SHOW:
            break
        correct = "✓" if check_correct(v['completion'], v['answer']) else "✗"
        print(f"[{correct}] completion: {v['completion']!r}  |  answer: {v['answer']!r}")

[✓] completion: '...the United States is the United States dollar.'  |  answer: 'Dollar'
[✓] completion: 'The official currency of the United States is the United States dollar (USD).'  |  answer: 'Dollar'
[✓] completion: 'The official currency of the United States is the United States dollar (USD).'  |  answer: 'Dollar'
[✗] completion: '...the Marshall Islands is the United States dollar.'  |  answer: 'Pound'
[✓] completion: 'The official currency of the United Kingdom is the Pound Sterling (GBP).'  |  answer: 'Pound'
[✓] completion: '...the United Kingdom is the Pound Sterling (GBP), commonly referred to as the Pound.'  |  answer: 'Pound'
[✓] completion: 'The United Kingdom is the Pound Sterling (GBP).'  |  answer: 'Pound'
[✓] completion: 'The official currency of Japan is the Japanese yen (¥).'  |  answer: 'Yen'
[✓] completion: 'Japan is the Japanese yen.'  |  answer: 'Yen'
[✓] completion: 'Japan is the Japanese yen (JPY).'  |  answer: 'Yen'
